# Centralized Value Learning

This notebook trains a model to predict the **value function** V(s) using:
- **TD(0)**: Bootstrap from target network
- **TD(λ)**: λ-returns with trajectory buffer

The ground truth comes from the analytical closed-form solution.

**Key differences from reward learning:**
- Discount γ > 0 (e.g., 0.99)
- Target network for bootstrapping
- Values are ~100x larger than rewards

In [1]:
# =============================================================================
# PARAMETERS
# =============================================================================

# --- Environment ---
WIDTH = 6
HEIGHT = 6
NUM_AGENTS = 1
NUM_APPLES = 5

# --- Value Learning ---
LEARNING_METHOD = "td_lambda"  # "td0" or "td_lambda"
DISCOUNT = 0.99
LAMBDA = 0.9  # Only used if LEARNING_METHOD == "td_lambda"

# --- Model ---
MODEL_TYPE = "CNN"  # "CNN" or "MLP"
MLP_HIDDEN_DIM = 64
MLP_NUM_LAYERS = 2
CNN_CONV_CHANNELS = [8]
CNN_KERNEL_SIZE = 3
CNN_HEAD_HIDDEN_DIM = 32
CNN_HEAD_NUM_LAYERS = 1

# --- Training ---
LR = 0.0001  # Lower LR for value learning
BATCH_SIZE = 64
MAX_STEPS = 200000
EVAL_FREQ = 1000
TARGET_UPDATE_FREQ = 1000
REPLAY_BUFFER_SIZE = 50000

# --- TD(λ) specific ---
TRAJECTORY_LENGTH = 100  # Steps per trajectory
NUM_TRAJECTORIES = 10    # Rolling Buffer Capacity
NUM_TRAJECTORIES_TO_TRAIN_ON = 5  # Number of trajectories to use for each training step

# --- Benchmarks ---
SOFT_THRESHOLD = 1  # Mean error < 1%
HARD_THRESHOLD = 1  # Max error < 1%
NUM_TEST_SAMPLES = 1000

# --- Output ---
EXPERIMENT_NAME = "value_cen"
OUTPUT_DIR = "outputs"
CSV_PATH = "outputs/experiment_results.csv"
SEED = 42

# --- Config ---
SAVE_MODELS = False

In [2]:
import sys
import time
import ast
from pathlib import Path
from typing import List, Optional

import numpy as np
import torch
from tqdm import tqdm

sys.path.append("../../")

# Environment & Helpers
from tadd_helpers.env_functions import State, init_empty_state
from tadd_helpers.setting_seed import set_all_seeds
from teleport_dynamic.base_value_model import BaseValueModelV2
from teleport_dynamic.model_cen_3ch import ValueCNNCentralized3Ch, ValueMLPCentralized3Ch
from teleport_dynamic.rewards_centralized import get_reward_centralized
from teleport_dynamic.orchard_generator import init_fixed_apples, teleport_agent

# Training functions
from teleport_dynamic.training_functions import (
    train_td0,
    train_td_lambda,
    TrajectoryBuffer,
)

# Experiment utilities
from teleport_dynamic.experiment_utils import (
    generate_centralized_value_test_set,
    evaluate_centralized_model,
    check_benchmark_centralized,
    append_experiment_result,
    format_eval_results_for_log,
    get_current_ram_mb
)
import os

if isinstance(CNN_CONV_CHANNELS, str):
    CNN_CONV_CHANNELS = ast.literal_eval(CNN_CONV_CHANNELS)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
set_all_seeds(SEED)

# Define Architecture String for Filename
if MODEL_TYPE == "CNN":
    chan_str = str(CNN_CONV_CHANNELS).replace(" ", "")
    arch_str = f"conv{chan_str}_k{CNN_KERNEL_SIZE}_head{CNN_HEAD_HIDDEN_DIM}x{CNN_HEAD_NUM_LAYERS}"
else:
    arch_str = f"mlp{MLP_HIDDEN_DIM}x{MLP_NUM_LAYERS}"

# Construct Descriptive Log Filename
pid = os.getpid()
method_str = LEARNING_METHOD if LEARNING_METHOD == "td0" else f"tdl{LAMBDA}"

# Create a unique string identifying this specific process run
run_id = f"{EXPERIMENT_NAME}_{method_str}_{MODEL_TYPE}_{WIDTH}x{HEIGHT}_{NUM_AGENTS}ag_g{DISCOUNT}_{arch_str}_pid{pid}"

log_filename = f"{run_id}.txt"

OUTPUT_DIR = Path(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOG_FILE = OUTPUT_DIR / log_filename

print(f"Run ID: {run_id}")
print(f"Logging to: {LOG_FILE}")

print(f"Device: {DEVICE}")
print(f"Learning Method: {LEARNING_METHOD}")
print(f"Discount (γ): {DISCOUNT}")
if LEARNING_METHOD == "td_lambda":
    print(f"Lambda (λ): {LAMBDA}")
print(f"Logging to: {LOG_FILE}")

🌟 Setting all seeds to 42...
✅ Seeds and deterministic settings configured.
Run ID: value_cen_tdl0.9_CNN_6x6_1ag_g0.99_conv[8]_k3_head32x1_pid57691
Logging to: outputs/value_cen_tdl0.9_CNN_6x6_1ag_g0.99_conv[8]_k3_head32x1_pid57691.txt
Device: cuda
Learning Method: td_lambda
Discount (γ): 0.99
Lambda (λ): 0.9
Logging to: outputs/value_cen_tdl0.9_CNN_6x6_1ag_g0.99_conv[8]_k3_head32x1_pid57691.txt


In [3]:
if SAVE_MODELS:
    save_dir = OUTPUT_DIR / "saved_models" / run_id
    save_dir.mkdir(parents=True, exist_ok=True)

In [4]:
# Model Initialization
model: BaseValueModelV2

if MODEL_TYPE == "CNN":
    model = ValueCNNCentralized3Ch(
        height=HEIGHT, width=WIDTH, lr=LR, discount=DISCOUNT,
        mlp_hidden_dim=CNN_HEAD_HIDDEN_DIM, mlp_num_layers=CNN_HEAD_NUM_LAYERS,
        conv_channels=CNN_CONV_CHANNELS, kernel_size=CNN_KERNEL_SIZE, 
        device=DEVICE, replay_buffer_capacity=REPLAY_BUFFER_SIZE
    )
    model_config = f"conv={CNN_CONV_CHANNELS},k={CNN_KERNEL_SIZE},head={CNN_HEAD_HIDDEN_DIM}"
else:
    model = ValueMLPCentralized3Ch(
        height=HEIGHT, width=WIDTH, lr=LR, discount=DISCOUNT,
        hidden_dim=MLP_HIDDEN_DIM, num_layers=MLP_NUM_LAYERS, 
        device=DEVICE, replay_buffer_capacity=REPLAY_BUFFER_SIZE
    )
    model_config = f"hidden={MLP_HIDDEN_DIM},layers={MLP_NUM_LAYERS}"

num_params = model.get_num_parameters()
print(f"Model: {MODEL_TYPE}")
print(f"Config: {model_config}")
print(f"Parameters: {num_params:,}")

Model: CNN
Config: conv=[8],k=3,head=32
Parameters: 9,505


In [5]:
# Initialize State and Test Set
state: State = init_fixed_apples(WIDTH, HEIGHT, NUM_AGENTS, NUM_APPLES)

print("Generating value test set (computing analytical solutions)...")
test_sets = generate_centralized_value_test_set(
    HEIGHT, WIDTH, NUM_AGENTS, NUM_APPLES, NUM_TEST_SAMPLES, 
    state.apples, DISCOUNT, SEED + 1000
)

# Show value statistics
all_values = []
for cases in test_sets.values():
    all_values.extend([c.true_value for c in cases])
all_values = np.array(all_values)

print(f"\nValue Statistics:")
print(f"  Min: {all_values.min():.4f}")
print(f"  Max: {all_values.max():.4f}")
print(f"  Mean: {all_values.mean():.4f}")
print(f"  Std: {all_values.std():.4f}")
print(f"\nTest set generated: {sum(len(v) for v in test_sets.values())} samples")

Generating value test set (computing analytical solutions)...

Value Statistics:
  Min: 13.7500
  Max: 14.7500
  Mean: 14.2500
  Std: 0.5000

Test set generated: 2000 samples


In [6]:
# Initialize trajectory buffer for TD(λ)
trajectory_buffer = None
if LEARNING_METHOD == "td_lambda":
    trajectory_buffer = TrajectoryBuffer(
        max_trajectories=NUM_TRAJECTORIES,
        max_trajectory_length=TRAJECTORY_LENGTH
    )
    print(f"Trajectory buffer initialized: {NUM_TRAJECTORIES} x {TRAJECTORY_LENGTH}")

Trajectory buffer initialized: 10 x 100


In [7]:
# Training Loop
with open(LOG_FILE, "w") as f:
    f.write(f"START Value Learning: {EXPERIMENT_NAME} | Method: {LEARNING_METHOD}\n")
    f.write(f"Grid: {WIDTH}x{HEIGHT}, Agents: {NUM_AGENTS}, Apples: {NUM_APPLES}\n")
    f.write(f"Discount: {DISCOUNT}, LR: {LR}\n")
    if LEARNING_METHOD == "td_lambda":
        f.write(f"Lambda: {LAMBDA}\n")
    f.write(f"Params: {num_params:,} | Config: {model_config}\n")
    f.write(f"Benchmarks: Soft < {SOFT_THRESHOLD}%, Hard < {HARD_THRESHOLD}%\n")
    f.write("="*80 + "\n")

soft_benchmark_step = None
hard_benchmark_step = None
last_loss = 0.0
start_time = time.time()
nan_detected = False

# Initialize c_t randomly
c_t = np.random.randint(0, NUM_AGENTS)
step = 0 
print(f"Starting {LEARNING_METHOD.upper()} training for {MAX_STEPS} steps...")
state: State = state # just unbound error is annoying
for step in range(MAX_STEPS):
    # Get reward (centralized: 1 if actor on apple, 0 otherwise)
    reward = get_reward_centralized(state, c_t)
    
    # Transition
    next_state = state.copy()
    teleport_agent(next_state, c_t)
    c_t_next = np.random.randint(0, NUM_AGENTS)
    
    # Add experience
    model.add_experience(state, next_state, reward, c_t, c_t_next)
    
    # Also add to trajectory buffer for TD(λ)
    if LEARNING_METHOD == "td_lambda" and trajectory_buffer is not None:
        s_enc = model.raw_state_to_nn_input(state, c_t)
        ns_enc = model.raw_state_to_nn_input(next_state, c_t_next)
        trajectory_buffer.add_step(s_enc, ns_enc, reward)
    
    # Train
    loss = None
    if LEARNING_METHOD == "td0":
        loss = train_td0(model, BATCH_SIZE)
    elif LEARNING_METHOD == "td_lambda" and trajectory_buffer is not None:
        if len(trajectory_buffer) >= NUM_TRAJECTORIES_TO_TRAIN_ON:
            loss = train_td_lambda(
                model, trajectory_buffer, DISCOUNT, LAMBDA, BATCH_SIZE
            )
    
    if loss is not None:
        last_loss = loss
        # NaN detection
        if np.isnan(loss) or np.isinf(loss):
            nan_detected = True
            print(f"\nNaN/Inf detected at step {step}! Stopping early.")
            with open(LOG_FILE, "a") as f:
                f.write(f"\nNaN/Inf detected at step {step}! Training stopped.\n")
            break
    
    # Update target network
    if step > 0 and step % TARGET_UPDATE_FREQ == 0:
        model.update_target_net()
    
    # Advance state
    state = next_state
    c_t = c_t_next
    
    # Evaluation
    if step > 0 and step % EVAL_FREQ == 0:
        results = evaluate_centralized_model(model, test_sets, use_value=True)
        soft, hard = check_benchmark_centralized(results, SOFT_THRESHOLD, HARD_THRESHOLD)
        ram_mb = get_current_ram_mb()
        
        if soft and soft_benchmark_step is None:
            soft_benchmark_step = step
            print(f"Soft benchmark achieved at step {step}!")
        if hard and hard_benchmark_step is None:
            hard_benchmark_step = step
            print(f"Hard benchmark achieved at step {step}!")
        
        bench_str = ""
        if hard_benchmark_step:
            bench_str = "HARD"
        elif soft_benchmark_step:
            bench_str = "Soft"
        else:
            bench_str = "-"
        
        if SAVE_MODELS:
            save_path = save_dir / f"{EXPERIMENT_NAME}_step_{step}.pt"
            save_path = save_dir / f"{EXPERIMENT_NAME}_step_{step}.pt"
            torch.save(model.state_dict(), save_path)
        
        log_line = format_eval_results_for_log(results, step, last_loss)
        log_line += f" | RAM: {ram_mb:.1f}MB | Bench: {bench_str}"
        
        with open(LOG_FILE, "a") as f:
            f.write(log_line + "\n")
        
        # Early stopping on hard benchmark
        if hard_benchmark_step is not None:
            break

# Final evaluation
wall_time = time.time() - start_time
final_results = evaluate_centralized_model(model, test_sets, use_value=True)

all_means = [r.mean_pct_error for r in final_results.values()]
all_maxes = [r.max_pct_error for r in final_results.values()]
final_mean = max(all_means) if all_means else float('nan')
final_max = max(all_maxes) if all_maxes else float('nan')

# Log to CSV
method_note = LEARNING_METHOD
if LEARNING_METHOD == "td_lambda":
    method_note += f"_l{LAMBDA}"
if nan_detected:
    method_note += "_NaN"

append_experiment_result(
    CSV_PATH, 
    f"value_{LEARNING_METHOD}", 
    MODEL_TYPE, 
    True,  # centralized
    f"{WIDTH}x{HEIGHT}", 
    NUM_AGENTS, 
    NUM_APPLES, 
    f"g{DISCOUNT}",
    model_config, 
    soft_benchmark_step, 
    hard_benchmark_step,
    step, 
    final_mean, 
    final_max, 
    wall_time, 
    num_parameters=model.get_num_parameters(),
    kernel_size=CNN_KERNEL_SIZE if MODEL_TYPE == "CNN" else None,
    learning_method=LEARNING_METHOD,  # "td0" or "td_lambda"
    notes=method_note
)

print(f"\nTraining complete!")
print(f"Final Mean Error: {final_mean:.2f}%")
print(f"Final Max Error: {final_max:.2f}%")
print(f"Wall Time: {wall_time:.1f}s")
print(f"Log saved to: {LOG_FILE}")

Starting TD_LAMBDA training for 200000 steps...


KeyboardInterrupt: 

In [ ]:
# Sanity Check: Visualize predictions vs analytical values
print("\n" + "="*80)
print("SANITY CHECK: Sample Predictions vs Analytical Values")
print("="*80)

for category, cases in test_sets.items():
    if not cases:
        continue
    
    print(f"\n>>> CATEGORY: {category.upper()} <<<")
    samples = cases[:3]
    
    for i, case in enumerate(samples):
        pred = model.get_value(case.state, acting_agent_idx=case.acting_agent_idx)
        actual = case.true_value
        assert actual is not None, "Analytical value should not be None in Value Learning"
        error = abs(pred - actual)
        pct_err = (error / abs(actual)) * 100 if abs(actual) > 1e-9 else error * 100
        
        print(f"\n--- Sample {i+1} ---")
        print(f"Acting Agent: {case.acting_agent_idx}")
        print(f"Immediate Reward: {case.true_reward}")
        print(f"PREDICTED V(s): {pred:.4f}")
        print(f"ANALYTICAL V(s): {actual:.4f}")
        print(f"ERROR: {error:.4f} ({pct_err:.2f}%)")
        print(case.state)


SANITY CHECK: Sample Predictions vs Analytical Values

>>> CATEGORY: PICKER <<<

--- Sample 1 ---
Acting Agent: 0
Immediate Reward: 1.0
PREDICTED V(s): 14.7605
ANALYTICAL V(s): 14.7500
ERROR: 0.0105 (0.07%)
--- Empty State (Grid: 6x6) ---

--- Agent Locations ---
  Agent 0: (5, 5)

--- Agents (Count) ---
. . . . . .
. . . . . .
. . . . . .
. . . . . .
. . . . . .
. . . . . 1

--- Apples (Count) ---
. . . . 1 .
. . . . . .
. 1 . . . 1
. . . . . .
. . . . . 1
. . . . . 1

--- Sample 2 ---
Acting Agent: 0
Immediate Reward: 1.0
PREDICTED V(s): 14.7612
ANALYTICAL V(s): 14.7500
ERROR: 0.0112 (0.08%)
--- Empty State (Grid: 6x6) ---

--- Agent Locations ---
  Agent 0: (0, 4)

--- Agents (Count) ---
. . . . 1 .
. . . . . .
. . . . . .
. . . . . .
. . . . . .
. . . . . .

--- Apples (Count) ---
. . . . 1 .
. . . . . .
. 1 . . . 1
. . . . . .
. . . . . 1
. . . . . 1

--- Sample 3 ---
Acting Agent: 0
Immediate Reward: 1.0
PREDICTED V(s): 14.7824
ANALYTICAL V(s): 14.7500
ERROR: 0.0324 (0.22%)
--- 